In [ ]:
import matplotlib.pyplot as plt
%matplotlib widget

from pprint import pprint
import scipy.io
import numpy as np
from pathlib import Path
import spikeinterface as si  # import core only
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.postprocessing as spost
import spikeinterface.qualitymetrics as sqm
import spikeinterface.comparison as sc
import spikeinterface.exporters as sexp
import spikeinterface.curation as scur
import spikeinterface.widgets as sw
from pathlib import Path
import re
import os
import probeinterface as pi
import probeinterface.plotting as pi_plot


global_job_kwargs = dict(n_jobs=4, chunk_duration="1s")
si.set_global_job_kwargs(**global_job_kwargs)


In [ ]:
recording= se.read_openephys(folder_path=r'Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28')
print(recording)
sorting_full = se.read_kilosort(r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4")
print(sorting_full)
trigger_data = scipy.io.loadmat(r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Triggers\mouse2_renewal_checkerboard_triggers.mat")


In [ ]:
triggers = trigger_data["evt"]
times = recording.get_times()

print(f"Start time in seconds: {times[0]}")


In [ ]:

# --- 1. CONFIGURATION ---
parent_folder = Path(r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data") 
target_folder_name = "mouse2_renewal_checkerboard_2025-06-12_12-52-28"

# --- 2. SORTING (Same as before) ---
date_pattern = re.compile(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})")

def get_datetime_from_name(folder_path):
    match = date_pattern.search(folder_path.name)
    if match:
        return match.group(1)
    return "9999"

all_folders = [f for f in parent_folder.iterdir() if f.is_dir()]
sorted_folders = sorted(all_folders, key=get_datetime_from_name)

# --- 3. ROBUST OFFSET CALCULATION ---
offset_samples = 0
found_target = False

print(f"Scanning {len(sorted_folders)} folders for OpenEphys data...")

for folder in sorted_folders:
    if folder.name == target_folder_name:
        found_target = True
        print(f" -> REACHED TARGET: {folder.name}")
        break 

    # --- RECURSIVE SEARCH FOR .oebin ---
    # We look for 'structure.oebin' anywhere inside this folder
    # This fixes the issue of nested "Record Node" folders
    oebin_files = list(folder.rglob("structure.oebin"))
    
    if len(oebin_files) == 0:
        # Fallback for older OpenEphys formats (binary.json)
        oebin_files = list(folder.rglob("binary.json"))

    if len(oebin_files) > 0:
        # We found the metadata file! Use its parent folder to load.
        # If multiple experiments exist in one session, this loads the first one found.
        recording_path = oebin_files[0].parent
        
        try:
            # Load the recording
            rec_temp = se.read_openephys(recording_path)
            
            # Calculate total samples (summing segments if necessary)
            n_samples = rec_temp.get_total_samples()
            offset_samples += n_samples
            
            print(f"   [+] Added {n_samples} samples from {folder.name}")
            print(f"       (Source: {recording_path.relative_to(parent_folder)})")
            
        except Exception as e:
            print(f"   [!] Failed to load {folder.name}: {e}")
    else:
        print(f"   [-] No OpenEphys data found in {folder.name}")

if found_target:
    print("="*40)
    print(f"FINAL OFFSET: {offset_samples} samples")
    print("="*40)
else:
    print("ERROR: Target folder not reached.")


In [ ]:
triggers_flat = np.array(triggers).flatten().astype(int)
triggers_post = triggers_flat - 87680117
print(triggers_flat)
print(triggers_post)


In [ ]:
final_rec = recording.frame_slice(triggers_post[0], triggers_post[-1])
final_sort = sorting_full.frame_slice(triggers_flat[0], triggers_flat[-1])

print(final_rec)
print(final_sort)


In [ ]:
print(f"Recording Channel IDs: {final_rec.get_channel_ids()[:10]}")
# If you have a specific property for channel indices, check that too
print(f"Sorting Unit IDs: {final_sort.get_unit_ids()[:10]}")

# Crucially, check if the sorting object has 'channel_ids' associated with it
# (Kilosort results usually do)
if final_sort.has_recording():
    print(f"Sorting internal recording channel IDs: {sorting._recording.get_channel_ids()[:10]}")

In [ ]:
sw.plot_probe_map(final_rec, with_channel_ids=True)


In [ ]:
# Create a lightweight analyzer for diagnosis
analyzer = si.create_sorting_analyzer(final_sort, final_rec, 
                                      format="memory", 
                                      sparse=True)

# Extract a small sample
analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=50)
analyzer.compute("waveforms")
analyzer.compute("templates")

# Plot the templates
sw.plot_unit_templates(analyzer, unit_ids=final_sort.get_unit_ids()[:5])


In [ ]:
# 1. Create a list of integers [0, 1, 2...] matching the number of channels
new_channel_ids = np.arange(final_rec.get_num_channels())

# 2. Rename the channels in the recording object
# This returns a new object 'final_rec_aligned' with the correct integer IDs
final_rec_aligned = final_rec.rename_channels(new_channel_ids)

print(f"Old IDs: {final_rec.get_channel_ids()[:5]}")
print(f"New IDs: {final_rec_aligned.get_channel_ids()[:5]}")


In [ ]:
analyzer = si.create_sorting_analyzer(
    sorting=final_sort, 
    recording=final_rec_aligned, 
    format="memory", 
    sparse=True
)

# 2. Extract waveforms
# Note: 'random_spikes' selects which spikes to look at
analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=50)
analyzer.compute("waveforms")
analyzer.compute("templates")


In [ ]:
# 3. Plot to verify
# You should now see physiological waveforms instead of flat lines
sw.plot_unit_templates(analyzer, unit_ids=final_sort.get_unit_ids()[:5])


In [ ]:
unit_to_plot = final_sort.get_unit_ids()[0]
print(f"Inspecting Unit: {unit_to_plot}")

# Corrected: Pass 'figsize' directly, not inside a dictionary
w = sw.plot_unit_templates(
    analyzer, 
    unit_ids=[unit_to_plot], 
    shade_templates=True,    
    backend="matplotlib",
    figsize=(15, 10)  # <--- Pass directly here
)

plt.show()

In [ ]:
# 1. Setup parameters
unit_id = final_sort.get_unit_ids()[0]
spike_frame = final_sort.get_unit_spike_train(unit_id)[10] # 10th spike
window_radius = 100 # frames (approx 3ms at 30kHz)

# 2. Manually fetch the raw data again
# We grab a window around the spike: [spike - 100 : spike + 100]
start_f = int(spike_frame - window_radius)
end_f = int(spike_frame + window_radius)

raw_traces = final_rec_aligned.get_traces(start_frame=start_f, end_frame=end_f)

# 3. Plot as a Heatmap (Channels vs Time)
plt.figure(figsize=(10, 12))

# We transpose so Time is X-axis, Channels are Y-axis
# vmin/vmax clip the noise so spikes stand out
plt.imshow(raw_traces.T, aspect='auto', cmap='RdBu', vmin=-100, vmax=100, origin='lower')

plt.title(f"Raw Data around Frame {spike_frame}\n(Red/Blue = Spiking Activity)")
plt.xlabel("Time (frames from window start)")
plt.ylabel("Channel ID")
plt.colorbar(label="Amplitude")

# Draw a line where the spike should be (center)
plt.axvline(x=window_radius, color='k', linestyle='--', label='Sorting Timestamp')
plt.legend()

plt.show()

In [ ]:
# 1. Check the length of your recording
total_samples = final_rec_aligned.get_total_samples()
print(f"Recording Total Samples: {total_samples}")

# 2. Check the timestamp of the spike we tried to plot
unit_id = final_sort.get_unit_ids()[0]
spike_frames = final_sort.get_unit_spike_train(unit_id)

# Grab the 10th spike again
if len(spike_frames) > 10:
    test_frame = spike_frames[10]
    print(f"Requested Spike Frame:   {test_frame}")
    
    # 3. The Truth Test
    if test_frame >= total_samples:
        print("\n[CRITICAL ISSUE FOUND]: The spike frame is OUTSIDE the recording limits.")
        print("Your sorting timestamps are likely still relative to the CONCATENATED start,")
        print("but your recording object has been reset to 0.")
    else:
        print("\n[OK]: The spike frame is within bounds.")
        
        # If it is within bounds, let's try to manually grab data to see if it returns anything
        trace_snippet = final_rec_aligned.get_traces(
            start_frame=int(test_frame), 
            end_frame=int(test_frame+100)
        )
        print(f"Manual Data Fetch Shape: {trace_snippet.shape}")
        if trace_snippet.size == 0:
            print("...but the data returned is empty! (Check file paths/slicing)")
else:
    print("Unit has too few spikes.")

In [ ]:
# 1. Point to the settings.xml FILE
settings_file = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28\Record Node 101\settings.xml"

# 2. Read the probe geometry
print(f"Loading probe from file: {settings_file}")
loaded_object = pi.read_openephys(settings_file)

# 3. Handle Single Probe vs ProbeGroup
if isinstance(loaded_object, pi.Probe):
    probe = loaded_object
else:
    probe = loaded_object.probes[0]

print(f"Probe has {probe.get_contact_count()} contacts.")

# 4. Wire the probe to your aligned recording
# Map the recording's clean IDs (0, 1, 2...) to the probe's contacts
probe.set_device_channel_indices(final_rec_aligned.get_channel_ids())

# 5. Attach the probe
final_rec_with_probe = final_rec_aligned.set_probe(probe)

plt.figure(figsize=(8, 12))

# Plot with contact IDs to verify your mapping
pi_plot.plot_probe(probe, with_contact_id=True, with_device_index=True)

plt.title("Interactive Probe Map (Zoom/Pan enabled)")
plt.show()

In [ ]:
# 7. Re-run the Analyzer with the specific IMRO geometry
print("Recalculating unit positions with IMRO geometry...")
# 1. Create the Analyzer
print("Initializing Analyzer...")
analyzer_imro = si.create_sorting_analyzer(
    sorting=final_sort, 
    recording=final_rec_with_probe, 
    format="memory", 
    sparse=True
)

# 2. Compute Metrics in the Correct Order
print("Computing Random Spikes...")
analyzer_imro.compute("random_spikes", method="uniform", max_spikes_per_unit=50)

print("Computing Waveforms...")
analyzer_imro.compute("waveforms")

print("Computing Templates...") # <--- This step is critical for locations
analyzer_imro.compute("templates")

print("Computing Noise Levels...")
analyzer_imro.compute("noise_levels")

print("Recalculating locations using Center of Mass (Robust Mode)...")

# We re-run ONLY the location step. The waveforms/templates are already in memory.
analyzer_imro.compute("unit_locations", method="center_of_mass")



In [ ]:
# Final Visualization
%matplotlib inline
plt.figure(figsize=(12, 8))

# Plot 1: Unit Locations
sw.plot_unit_locations(analyzer_imro, backend="matplotlib")
plt.title("Unit Locations (Center of Mass)")
plt.show()

# Plot 2: Waveforms (To verify data exists)
sw.plot_unit_templates(analyzer_imro, unit_ids=final_sort.get_unit_ids()[:5], backend="matplotlib")
plt.title("Waveforms")
plt.show()


In [ ]:
# 1. Check Probe Geometry
print("--- 1. PROBE GEOMETRY ---")
positions = probe.contact_positions
print(f"Probe has {len(positions)} contacts.")
print(f"Sample positions (x, y): \n{positions[:5]}")
if np.isnan(positions).any():
    print("CRITICAL: Probe positions contain NaNs!")
if np.all(positions == 0):
    print("CRITICAL: All Probe positions are Zero!")

# 2. Check Templates
print("\n--- 2. TEMPLATES ---")
# Get templates data: (num_units, num_samples, num_channels)
templates = analyzer_imro.get_extension("templates").get_data()
print(f"Templates Shape: {templates.shape}")

# Check max amplitude of the first unit
max_amp = np.max(np.abs(templates[0]))
print(f"Max Amplitude of Unit 0: {max_amp}")

if max_amp == 0:
    print("CRITICAL: Template is FLAT (all zeros). Channel mapping is still wrong.")

# 3. Check Calculated Locations
print("\n--- 3. UNIT LOCATIONS ---")
locs = analyzer_imro.get_extension("unit_locations").get_data()
print(f"Locations Shape: {locs.shape}")
print(f"Sample Locations (x, y): \n{locs[:5]}")

if np.isnan(locs).any():
    print("CRITICAL: Calculated locations contain NaNs (Math failed).")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Get the raw location data
locs = analyzer_imro.get_extension("unit_locations").get_data()
unit_ids = final_sort.get_unit_ids()

# 2. Create a "Valid" mask (True if x is not NaN)
valid_mask = ~np.isnan(locs[:, 0])
valid_locs = locs[valid_mask]
valid_ids = unit_ids[valid_mask]

print(f"Total Units: {len(locs)}")
print(f"Valid Units: {len(valid_locs)}")
print(f"Dropped {len(locs) - len(valid_locs)} units due to calculation errors.")

# 3. Manual Plotting
fig, ax = plt.subplots(figsize=(10, 15))

# Plot the PROBE (Grey Dots)
# We get probe contacts from the object attached to the recording
probegroup = final_rec_with_probe.get_probegroup()
probe_2d = probegroup.probes[0]
contact_positions = probe_2d.contact_positions

ax.scatter(contact_positions[:, 0], contact_positions[:, 1], 
           s=20, color='grey', alpha=0.3, label='Probe Contacts')

# Plot the UNITS (Colored Dots)
# We use a scatter plot. C=unit_index can color them randomly/by depth
ax.scatter(valid_locs[:, 0], valid_locs[:, 1], 
           s=50, alpha=0.8, c='blue', edgecolors='white', label='Units')

ax.set_title(f"Neuropixels Map\n({len(valid_locs)} units)")
ax.set_xlabel("Width (um)")
ax.set_ylabel("Depth (um)")
ax.legend()
ax.axis('equal') # Important so the probe looks like a stick, not a square
plt.show()

In [ ]:
# 1. Get Unit Depths (Y-coordinates)
# locs is (num_units, 2) -> we want column 1 (y-axis)
locs = analyzer_imro.get_extension("unit_locations").get_data()
unit_depths = locs[:, 1] 

# 2. Get All Spikes efficiently
# This returns a structured array with 'sample_index' and 'unit_index' for every spike
all_spikes = final_sort.to_spike_vector()
spike_frames = all_spikes['sample_index']
spike_unit_indices = all_spikes['unit_index']

# 3. Map Spikes to Depths
# We use numpy array indexing to assign a depth to every single spike based on its unit
spike_depths = unit_depths[spike_unit_indices]

# 4. Filter out the "NaN" units (the ones that failed calculation earlier)
valid_mask = ~np.isnan(spike_depths)

# Convert frames to seconds
fs = final_rec_aligned.get_sampling_frequency()
clean_times = spike_frames[valid_mask] / fs
clean_depths = spike_depths[valid_mask]

# 5. Plot
plt.figure(figsize=(20, 10))

# s=1 (small dots), alpha=0.1 (transparency) helps visualize density
plt.scatter(clean_times, clean_depths, s=1, c='black', alpha=0.1)

plt.title(f"Spikes over Time and Depth ({len(clean_times)} spikes)")
plt.xlabel("Time (s)")
plt.ylabel("Depth (um)")

# Optional: Set Y-axis limits to match your probe geometry
# plt.ylim(0, 3840) 

plt.show()


In [ ]:
import numpy as np

# Get the Y positions (depths) of all electrode contacts
probegroup = final_rec_with_probe.get_probegroup()
probe_2d = probegroup.probes[0]
y_coords = probe_2d.contact_positions[:, 1]

print(f"Min Depth: {np.min(y_coords)}")
print(f"Max Depth: {np.max(y_coords)}")
print(f"Unique Depths: {len(np.unique(y_coords))}")

if len(np.unique(y_coords)) < 50:
    print("DIAGNOSIS: The probe geometry is FLATTENED. 'settings.xml' likely contains bad coordinates.")
else:
    print("DIAGNOSIS: The probe geometry is GOOD (spread out). The issue is in the signal.")

In [ ]:
import matplotlib.pyplot as plt

# 1. Fetch 100ms of raw data (approx 3000 samples)
# We choose a segment slightly into the recording to avoid startup artifacts
data_chunk = final_rec_with_probe.get_traces(start_frame=30000, end_frame=33000)

# 2. Get Channel Depths
y_coords = final_rec_with_probe.get_probe().contact_positions[:, 1]

# 3. Sort channels by depth (Deepest first)
sorted_indices = np.argsort(y_coords)
sorted_data = data_chunk[:, sorted_indices]
sorted_depths = y_coords[sorted_indices]

# 4. Plot Heatmap
plt.figure(figsize=(12, 8))

# Transpose so Time is X, Depth is Y
# vmin/vmax are set to +/- 50 microvolts to make spikes visible
plt.imshow(sorted_data.T, aspect='auto', cmap='RdBu', vmin=-50, vmax=50, 
           origin='lower', extent=[0, 100, np.min(y_coords), np.max(y_coords)])

plt.title("Raw Data Ordered by Depth")
plt.xlabel("Time (ms)")
plt.ylabel("Depth (um)")
plt.colorbar(label="Amplitude (uV)")
plt.show()

In [ ]:
# 1. Look at the ORIGINAL channel IDs (before we renamed them)
# We can access them if you still have 'final_rec' or by reloading briefly
original_ids = final_rec.get_channel_ids() 

print(f"First 10 IDs: {original_ids[:10]}")
print(f"Last 10 IDs:  {original_ids[-10:]}")

# Check if they are sorted
# If they look like ['CH1', 'CH3', 'CH2'...] then our simple range(0, N) rename corrupted the map.
is_sorted = True
try:
    # Extract numbers from strings like 'CH101' -> 101
    import re
    nums = [int(re.search(r'\d+', str(x)).group()) for x in original_ids]
    if nums != sorted(nums):
        is_sorted = False
        print("\n[CRITICAL]: The original channels were NOT in order!")
        print("Renaming them blindly to 0, 1, 2... scrambled the map.")
    else:
        print("\n[OK]: The original channels seem sorted.")
except:
    print("\nCould not parse IDs automatically. Please inspect visually.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Get the max channel for every unit from the templates we computed
templates = analyzer_imro.get_extension("templates").get_data()
# templates shape: (num_units, num_samples, num_channels)

# Find the channel index with the peak amplitude for each unit
max_channels = np.argmax(np.max(np.abs(templates), axis=1), axis=1)

# 2. Plot
plt.figure(figsize=(10, 5))
plt.hist(max_channels, bins=50, color='purple', alpha=0.7)
plt.title("Distribution of Unit Peak Channels")
plt.xlabel("Channel Index (0-383)")
plt.ylabel("Count of Units")
plt.show()

In [ ]:
import spikeinterface as si
import spikeinterface.widgets as sw
import matplotlib.pyplot as plt

print("1. Running Analyzer in DENSE mode (Scanning all channels)...")

# CRITICAL CHANGE: sparse=False
# This ignores Kilosort's initial channel guess and looks everywhere.
analyzer_dense = si.create_sorting_analyzer(
    sorting=final_sort, 
    recording=final_rec_with_probe, 
    format="memory", 
    sparse=False
)

# 2. Compute Waveforms & Templates
# Now we extract the full 384-channel footprint for every unit
analyzer_dense.compute("random_spikes", method="uniform", max_spikes_per_unit=50)
analyzer_dense.compute("waveforms")
analyzer_dense.compute("templates")
analyzer_dense.compute("noise_levels")

print("Templates computed. Re-calculating locations...")

# 3. Compute Locations using Center of Mass
# (CoM is safer than monopolar for now)
analyzer_dense.compute("unit_locations", method="center_of_mass")

# 4. Final Plot
%matplotlib inline
plt.figure(figsize=(12, 8))

# Plot 1: The Drift Map (Depth vs Time)
# This should now show lines distributed all across the Y-axis
sw.plot_unit_locations(analyzer_dense, backend="matplotlib")
plt.title("Corrected Unit Locations (Dense Scan)")
plt.show()

# Plot 2: Check the new Peak Channel Distribution
# If this works, this histogram will now be spread out!
templates = analyzer_dense.get_extension("templates").get_data()
max_channels = np.argmax(np.max(np.abs(templates), axis=1), axis=1)

plt.figure(figsize=(10, 4))
plt.hist(max_channels, bins=50, color='green', alpha=0.7)
plt.title("New Distribution of Peak Channels (Should be spread out)")
plt.show()

In [ ]:
import numpy as np
import os

# 1. Point this to the folder containing your 'spike_times.npy' etc.
kilosort_folder = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

# 2. Load the channel map
try:
    channel_map = np.load(os.path.join(kilosort_folder, 'channel_map.npy'))
    print(f"Kilosort used {len(channel_map)} channels.")
    print(f"First 10 mapped indices: {channel_map.flatten()[:10]}")
    
    # Check if it's just a simple 0..N range
    expected = np.arange(len(channel_map))
    if np.array_equal(channel_map.flatten(), expected):
        print("Map is linear (0, 1, 2...). Kilosort used all channels in order.")
    else:
        print("Map is NON-LINEAR or SUBSET! Kilosort skipped or reordered channels.")
        print(f"Example weirdness: {channel_map.flatten()[100:110]}")
        
except FileNotFoundError:
    print("Could not find channel_map.npy. Assuming linear map 0-383.")

In [ ]:
import matplotlib.pyplot as plt

# 1. Get a chunk of raw data (e.g. 1 second)
# Ensure we use the recording that has the probe attached
raw_chunk = final_rec_with_probe.get_traces(start_frame=10000, end_frame=40000)

# 2. Extract specific channels
ch_275 = raw_chunk[:, 275]
ch_0 = raw_chunk[:, 0]

# 3. Plot comparison
plt.figure(figsize=(15, 6))

plt.subplot(2, 1, 1)
plt.plot(ch_275, color='red')
plt.title("Channel 275 (Suspected Artifact)")

plt.subplot(2, 1, 2)
plt.plot(ch_0, color='blue')
plt.title("Channel 0 (Comparison)")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import spikeinterface as si
import spikeinterface.widgets as sw
import matplotlib.pyplot as plt
import probeinterface as pi
import os

# --- PATHS ---
# Update these to your actual locations
settings_file = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28\Record Node 101\settings.xml"

# 1. LOAD MAP & FLATTEN
channel_map = np.load(os.path.join(kilosort_folder, 'channel_map.npy')).flatten()
print(f"Loaded Kilosort Map: {len(channel_map)} channels")
print(f"First 5 indices: {channel_map[:5]}")

# 2. PREPARE THE FULL RECORDING (Linear IDs 0..N)
# We go back to your 'final_rec' (assuming it has the full channel count)
# We rename it 0..N to make the mapping simple
full_ids = np.arange(final_rec.get_num_channels())
rec_full = final_rec.rename_channels(full_ids)

# 3. ATTACH PROBE TO FULL RECORDING
# We attach the probe BEFORE slicing, so SI automatically handles the geometry slicing
probe_group = pi.read_openephys(settings_file)
if isinstance(probe_group, pi.Probe):
    probe = probe_group
else:
    probe = probe_group.probes[0]
    
# Map probe to the full linear IDs
probe.set_device_channel_indices(full_ids)
rec_full = rec_full.set_probe(probe)

# 4. SLICE & REORDER (The Critical Fix)
# This creates a new recording where "Channel 0" is actually "Original Channel 48"
# matching Kilosort's reality exactly.
rec_kilosort_ordered = rec_full.select_channels(channel_map)

print(f"Reordered Recording Channels: {rec_kilosort_ordered.get_num_channels()}")

In [ ]:
# 5. RUN ANALYZER
print("Running Analyzer on Kilosort-Aligned Data...")

# We can now safely use sparse=True because the alignment is correct
analyzer_fixed = si.create_sorting_analyzer(
    sorting=final_sort, 
    recording=rec_kilosort_ordered, 
    format="memory", 
    sparse=True 
)

analyzer_fixed.compute("random_spikes", method="uniform", max_spikes_per_unit=50)
analyzer_fixed.compute("waveforms")
analyzer_fixed.compute("templates")
analyzer_fixed.compute("noise_levels")

# Use Center of Mass for robustness
analyzer_fixed.compute("unit_locations", method="center_of_mass")

print("Analysis Complete.")

# 6. FINAL PLOTS
%matplotlib inline 

# Plot 1: Drift Map (Should be distributed!)
sw.plot_unit_locations(analyzer_fixed, backend="matplotlib")
plt.title("Drift Map (Corrected Channel Map)")
plt.show()

# Plot 2: Waveforms (Should correspond to real spikes now)
sw.plot_unit_templates(analyzer_fixed, unit_ids=final_sort.get_unit_ids()[:5], backend="matplotlib")
plt.title("Waveforms")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Get the first unit's ID and its spikes
unit_id = final_sort.get_unit_ids()[0]
spikes = final_sort.get_unit_spike_train(unit_id)
first_spike_frame = spikes[0]

# 2. Extract raw data around this spike ( +/- 3ms)
# We use 'rec_kilosort_ordered' (the one we just fixed with the channel map)
sampling_rate = rec_kilosort_ordered.get_sampling_frequency()
window = int(3e-3 * sampling_rate) # 3ms window
start = int(first_spike_frame - window)
end = int(first_spike_frame + window)

print(f"Checking Unit {unit_id}")
print(f"Spike Frame: {first_spike_frame}")
print(f"Extracting Raw Data: {start} to {end}")

# Safety check for duration
if start < 0 or end > rec_kilosort_ordered.get_num_frames():
    print("CRITICAL: Spike time is OUTSIDE recording duration!")
else:
    raw_snippet = rec_kilosort_ordered.get_traces(start_frame=start, end_frame=end)
    
    # 3. Plot ALL channels to see if the spike exists ANYWHERE
    plt.figure(figsize=(10, 6))
    plt.imshow(raw_snippet.T, aspect='auto', vmin=-100, vmax=100, cmap='RdBu')
    plt.colorbar(label="Amplitude (uV)")
    plt.title(f"Raw Data at Spike Time (Frame {first_spike_frame})")
    plt.xlabel("Time (samples)")
    plt.ylabel("Channel Index (0-383)")
    
    # Draw a line where the spike SHOULD be (center)
    plt.axvline(x=window, color='green', linestyle='--', label='Kilosort Timestamp')
    plt.legend()
    plt.show()

In [ ]:
import spikeinterface as si
import spikeinterface.extractors as se
import numpy as np
import os
import matplotlib.pyplot as plt

# --- 1. SETUP PATHS ---
raw_data_folder = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28\Record Node 101"
kilosort_folder = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

# --- 2. DEFINE SLICING INDICES (LOGIC CORRECTION) ---

# We define the margins
margin = 100 

# REVERTED LOGIC:
# rec_raw is a single session (Local Time) -> Needs triggers_flat
rec_start_frame = int(triggers_flat[0]) - margin
rec_end_frame   = int(triggers_flat[-1]) + margin

# final_sort is concatenated (Global Time) -> Needs triggers_post
sort_start_frame = int(triggers_post[0]) - margin
sort_end_frame   = int(triggers_post[-1]) + margin

print("--- DIAGNOSTIC CHECK ---")
print(f"Recording Slice (Local): {rec_start_frame} to {rec_end_frame}")
print(f"Sorting Slice   (Global): {sort_start_frame} to {sort_end_frame}")

# Check where the spikes actually are
first_unit = final_sort.get_unit_ids()[0]
spikes = final_sort.get_unit_spike_train(first_unit)
print(f"Sample Spikes (Global): {spikes[:5]} ... {spikes[-5:]}")

if spikes[0] > sort_end_frame or spikes[-1] < sort_start_frame:
    print("WARNING: The requested slice window DOES NOT OVERLAP with the spike times!")
else:
    print("SUCCESS: Spike times fall within the Global Slice window.")
print("------------------------")


# --- 3. PREPARE RECORDING (Rename -> Map -> Slice) ---

# A. Load Raw Data
rec_raw = se.read_openephys(raw_data_folder)

# B. FIX: Rename channels to Integers (0..N) so they match Kilosort map
num_channels = rec_raw.get_num_channels()
rec_raw = rec_raw.rename_channels(np.arange(num_channels))

# C. Apply Kilosort Channel Map
channel_map = np.load(os.path.join(kilosort_folder, 'channel_map.npy')).flatten()
rec_ordered = rec_raw.select_channels(channel_map)

# D. Slice the Recording
# Ensure we don't go below 0
start_safe_rec = max(0, rec_start_frame)
rec_synced = rec_ordered.frame_slice(start_frame=start_safe_rec, end_frame=rec_end_frame)

print(f"Synced Recording Duration: {rec_synced.get_num_frames()} frames")


# --- 4. PREPARE SORTING (Slice + Shift) ---

# Slice the sorting using GLOBAL triggers
start_safe_sort = max(0, sort_start_frame)
sort_synced = final_sort.frame_slice(start_frame=start_safe_sort, end_frame=sort_end_frame)

print(f"Synced Sorting Units: {len(sort_synced.get_unit_ids())}")

# Double check we didn't slice it to empty
if sort_synced.get_total_num_spikes() == 0:
    raise ValueError("CRITICAL: The sliced sorting object is EMPTY. Check 'sort_start_frame' vs spike times.")


# --- 5. VERIFY ALIGNMENT (The "Blob" Check) ---

unit_ids = sort_synced.get_unit_ids()
if len(unit_ids) > 0:
    first_spike = None
    # Find a unit that actually has spikes in this segment
    for uid in unit_ids:
        spikes = sort_synced.get_unit_spike_train(uid)
        if len(spikes) > 0:
            first_spike = spikes[0]
            break
    
    if first_spike is not None:
        # Extract raw data at this time from the SYNCED recording
        window = 100 # samples
        start = max(0, first_spike - window)
        end = first_spike + window
        
        # Check bounds
        if end < rec_synced.get_num_frames():
            snippet = rec_synced.get_traces(start_frame=start, end_frame=end)
            
            plt.figure(figsize=(8, 5))
            plt.imshow(snippet.T, aspect='auto', cmap='RdBu', vmin=-100, vmax=100, origin='lower')
            plt.axvline(x=window, color='green', linestyle='--', label='Aligned Spike Time')
            plt.title("Alignment Verification\n(Blob should be at green line)")
            plt.legend()
            plt.show()
        else:
            print("Warning: First spike is too close to end of recording.")
    else:
        print("Warning: No spikes found in this segment for any unit.")
else:
    print("Warning: No units found in this segment.")


# --- 6. RUN ANALYZER ---
print("Running Analyzer...")

analyzer = si.create_sorting_analyzer(
    sorting=sort_synced, 
    recording=rec_synced, 
    format="memory", 
    sparse=True 
)

analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=50)
analyzer.compute("waveforms")
analyzer.compute("templates")
analyzer.compute("noise_levels")
analyzer.compute("unit_locations", method="center_of_mass")

import spikeinterface.widgets as sw
sw.plot_unit_locations(analyzer, backend="matplotlib")
plt.title("Corrected Unit Locations")
plt.show()


In [ ]:
final_rec = recording.frame_slice(triggers_post[1], triggers_post[-1])
print(final_rec.get_total_samples())
print(len(triggers_flat))

print((final_rec.get_total_samples()+(2 * 30000))/(len(triggers_flat)-1))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import spikeinterface.extractors as se

# 1. Define the Kilosort Folder
kilosort_folder = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

print(f"Loading Kilosort data from: {kilosort_folder}")

# 2. Load the Sorting
# This loads spike_times.npy and spike_clusters.npy
sorting_ks = se.read_kilosort(folder_path=kilosort_folder)

# 3. Get Unit Depths (The robust way)
# We will read the Kilosort metadata directly to find where each unit is located.
print("Calculating unit depths from Kilosort templates...")

# A. Load Channel Positions (X, Y)
chan_pos_path = os.path.join(kilosort_folder, 'channel_positions.npy')
if not os.path.exists(chan_pos_path):
    raise FileNotFoundError("Could not find channel_positions.npy in the Kilosort folder.")

channel_positions = np.load(chan_pos_path)
# We want the Y-coordinate (Depth) -> Column 1
channel_depths = channel_positions[:, 1]

# B. Load Templates (or create a mapping if templates are missing)
# Kilosort 4 usually saves 'templates.npy' or 'stat.npy'.
# SpikeInterface can usually infer the 'main_channel' property automatically.

unit_depth_map = {}

# Check if SpikeInterface already loaded the properties
if sorting_ks.get_property('y') is not None:
    # Use existing depths
    depths = sorting_ks.get_property('y')
    ids = sorting_ks.unit_ids
    unit_depth_map = dict(zip(ids, depths))
    print("Loaded pre-calculated depths from metadata.")
    
else:
    # Fallback: Calculate peak channel manually
    print("Metadata missing. Calculating depths from template peaks...")
    
    # Try loading templates
    temp_path = os.path.join(kilosort_folder, 'templates.npy')
    
    if os.path.exists(temp_path):
        templates = np.load(temp_path)
        # templates shape: (n_units, n_timepoints, n_channels)
        
        # Find the channel with the max amplitude for each unit
        # (We use Peak-to-Peak amplitude)
        ptp = np.ptp(templates, axis=1) 
        peak_channels = np.argmax(ptp, axis=1)
        
        # Map Unit Index -> Depth
        for i, unit_id in enumerate(sorting_ks.unit_ids):
            # Safety check if Kilosort removed some units
            if i < len(peak_channels):
                chan_idx = peak_channels[i]
                unit_depth_map[unit_id] = channel_depths[chan_idx]
    else:
        print("Warning: could not find templates.npy. Using 'cluster_info.tsv' if available.")
        # Final fallback: cluster_info.tsv usually has 'depth' column
        import pandas as pd
        info_path = os.path.join(kilosort_folder, 'cluster_info.tsv')
        if os.path.exists(info_path):
            df = pd.read_csv(info_path, sep='\t')
            for _, row in df.iterrows():
                unit_depth_map[row['cluster_id']] = row['depth']


# 4. Prepare Plot Data
# --------------------
print("Generating spike vectors...")
all_spikes = sorting_ks.to_spike_vector()
spike_times = all_spikes['sample_index']
spike_unit_indices = all_spikes['unit_index']
unit_ids = sorting_ks.unit_ids

# Map indices to depths
# Create a lookup array for fast indexing
max_idx = np.max(spike_unit_indices)
index_to_depth = np.full(max_idx + 1, np.nan)

for i, unit_id in enumerate(unit_ids):
    if unit_id in unit_depth_map:
        # We need to map the 'unit index' (0, 1, 2...) to the depth
        # spike_vector uses the index of the unit in the unit_ids list
        index_to_depth[i] = unit_depth_map[unit_id]

spike_depths = index_to_depth[spike_unit_indices]

# Filter NaNs
valid_mask = ~np.isnan(spike_depths)
plot_times = spike_times[valid_mask]
plot_depths = spike_depths[valid_mask]

# 5. Plot
# -------
print(f"Plotting {len(plot_times)} spikes...")

plt.figure(figsize=(15, 10))

# Rasterized=True makes saving/rendering much faster for millions of dots
plt.scatter(plot_times, plot_depths, s=0.5, c='k', alpha=0.1, rasterized=True)

plt.title("Kilosort Drift Map (Concatenated)")
plt.xlabel("Global Time (Frames)")
plt.ylabel("Depth (um)")
plt.margins(x=0)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import spikeinterface.extractors as se

# 1. Define Paths and Range
kilosort_folder = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

# Use the Global Triggers (triggers_post) to define the window
# We assume triggers_post is available in your workspace
t_start = float(triggers_post[0])
t_end = float(triggers_post[-1])
buffer = 1000 # Add a small buffer (in frames) to see just past the edges

print(f"Plotting Frame: {t_start} to {t_end}")
print(f"Total Duration: {(t_end - t_start) / 30000:.2f} seconds (assuming 30kHz)")

# 2. Load Kilosort Data
print("Loading Kilosort data...")
sorting_ks = se.read_kilosort(folder_path=kilosort_folder)

# 3. Get Unit Depths (Robust Method)
print("Calculating unit depths...")
chan_pos_path = os.path.join(kilosort_folder, 'channel_positions.npy')
channel_positions = np.load(chan_pos_path)
channel_depths = channel_positions[:, 1]

unit_depth_map = {}

# Try to use cluster_info.tsv first (most accurate)
info_path = os.path.join(kilosort_folder, 'cluster_info.tsv')
if os.path.exists(info_path):
    import pandas as pd
    df = pd.read_csv(info_path, sep='\t')
    for _, row in df.iterrows():
        unit_depth_map[row['cluster_id']] = row['depth']
else:
    # Fallback: Estimate from templates
    print("cluster_info.tsv not found, estimating from templates...")
    temp_path = os.path.join(kilosort_folder, 'templates.npy')
    if os.path.exists(temp_path):
        templates = np.load(temp_path)
        ptp = np.ptp(templates, axis=1)
        peak_channels = np.argmax(ptp, axis=1)
        for i, unit_id in enumerate(sorting_ks.unit_ids):
            if i < len(peak_channels):
                unit_depth_map[unit_id] = channel_depths[peak_channels[i]]

# 4. Filter Spikes to Window
print("Filtering spikes...")
all_spikes = sorting_ks.to_spike_vector()
spike_times = all_spikes['sample_index']
spike_unit_indices = all_spikes['unit_index']
unit_ids = sorting_ks.unit_ids

# A. Filter by Time FIRST (faster)
# Keep only spikes within the trigger frame
time_mask = (spike_times >= (t_start - buffer)) & (spike_times <= (t_end + buffer))
valid_times = spike_times[time_mask]
valid_indices = spike_unit_indices[time_mask]

# B. Map to Depths
# Create lookup array for fast mapping
max_idx = np.max(spike_unit_indices)
index_to_depth = np.full(max_idx + 1, np.nan)

for i, unit_id in enumerate(unit_ids):
    if unit_id in unit_depth_map:
        index_to_depth[i] = unit_depth_map[unit_id]

valid_depths = index_to_depth[valid_indices]

# C. Remove units with no depth
depth_mask = ~np.isnan(valid_depths)
plot_times = valid_times[depth_mask]
plot_depths = valid_depths[depth_mask]

# 5. Plot
print(f"Rendering {len(plot_times)} spikes...")
plt.figure(figsize=(16, 8))

# Scatter Spikes
plt.scatter(plot_times, plot_depths, s=0.5, c='black', alpha=0.3, rasterized=True, label='Spikes')

# Plot Vertical Lines for ALL Triggers
# This visualizes the structure of your task
for i, trig in enumerate(triggers_post):
    # Only label the first and last to keep legend clean
    label = 'Triggers' if i == 0 else None 
    color = 'green' if i == 0 else ('red' if i == len(triggers_post)-1 else 'orange')
    alpha = 1.0 if (i == 0 or i == len(triggers_post)-1) else 0.5
    plt.axvline(x=trig, color=color, linestyle='--', alpha=alpha, linewidth=1, label=label)

plt.xlim(t_start - buffer, t_end + buffer)
plt.title("Kilosort Drift Map (Windowed by Triggers)")
plt.xlabel("Global Time (Frames)")
plt.ylabel("Depth (um)")
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# --- RUN THIS TO GET A WORKING ANALYZER ---
# This combines the "Blob Check" success with the Analyzer

# 1. SETUP (As verified)
# ----------------------
raw_data_folder = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28"
rec_raw = se.read_openephys(raw_data_folder)
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels())) # Fix string IDs

# Apply Map
channel_map = np.load(os.path.join(kilosort_folder, 'channel_map.npy')).flatten()
rec_ordered = rec_raw.select_channels(channel_map)

# Slice Recording (Local Triggers)
rec_start = int(triggers_post[0]) - 100
rec_end   = int(triggers_post[-1]) + 100
rec_synced = rec_ordered.frame_slice(rec_start, rec_end)

# Slice Sorting (Global Triggers)
sort_start = int(triggers_flat[0]) - 100
sort_end   = int(triggers_flat[-1]) + 100
sort_synced = sorting_ks.frame_slice(sort_start, sort_end)

# 2. RUN ANALYZER
# ----------------------
# We use sparse=True because we now trust the channel map is correct.
analyzer = si.create_sorting_analyzer(
    sorting=sort_synced, 
    recording=rec_synced, 
    format="memory", 
    sparse=True 
)

# These steps re-extract the data. 
# If alignment is correct, 'waveforms' will capture real spikes, 
# and 'unit_locations' will calculate the correct depths.
analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=50)
analyzer.compute("waveforms")
analyzer.compute("templates")
analyzer.compute("unit_locations", method="center_of_mass")

# 3. PLOT
# ----------------------
import spikeinterface.widgets as sw
sw.plot_unit_locations(analyzer, backend="matplotlib")
plt.title("Analyzer Plot (Should now match Manual Plot)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Get Calculated Depths (from the Analyzer's own math)
# -------------------------------------------------------
# These are the depths computed from the waveforms extracted from 'rec_synced'
locs = analyzer.get_extension("unit_locations").get_data()
unit_depths = locs[:, 1] # Y-coordinates

# DIAGNOSTIC: Check if depths are clumped
depth_range = np.ptp(unit_depths) # Peak-to-peak (Max - Min)
print(f"Depth Range: {np.min(unit_depths):.1f} um to {np.max(unit_depths):.1f} um")

if depth_range < 50:
    print("WARNING: All units are clumped! Alignment/Channel Map is still broken.")
else:
    print("SUCCESS: Units are distributed across the probe.")

# 2. Get Spike Times (from the Sliced Sorting)
# -------------------------------------------------------
sorting = analyzer.sorting
all_spikes = sorting.to_spike_vector()
spike_times = all_spikes['sample_index']
spike_unit_indices = all_spikes['unit_index']

# 3. Map Spikes to Depths
# -------------------------------------------------------
# Create a lookup array: Index -> Depth
spike_depths = unit_depths[spike_unit_indices]

# Filter out NaNs (failed units)
valid_mask = ~np.isnan(spike_depths)
plot_times = spike_times[valid_mask]
plot_depths = spike_depths[valid_mask]

# Convert frames to seconds
fs = analyzer.recording.get_sampling_frequency()
plot_times_sec = plot_times / fs

# 4. Plot
# -------------------------------------------------------
plt.figure(figsize=(12, 8))

# s=1, alpha=0.1 allows you to see density
plt.scatter(plot_times_sec, plot_depths, s=1, c='black', alpha=0.1, rasterized=True)

plt.title(f"Analyzer Drift Map\n({len(plot_times)} spikes from {len(unit_depths)} units)")
plt.xlabel("Time (seconds)")
plt.ylabel("Depth (um)")

# Optional: Set limits to match probe size
# plt.ylim(0, 3840) 

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import spikeinterface.widgets as sw

# 1. Get the "Good" Depths from Kilosort Metadata
# (We repeat the logic from the manual plot that worked)
print("Retrieving correct depths from Kilosort...")

unit_depth_map = {}

# Try cluster_info first (Best source)
info_path = os.path.join(kilosort_folder, 'cluster_info.tsv')
if os.path.exists(info_path):
    import pandas as pd
    df = pd.read_csv(info_path, sep='\t')
    for _, row in df.iterrows():
        unit_depth_map[row['cluster_id']] = row['depth']
else:
    # Fallback to templates
    print("Estimating from templates...")
    temp_path = os.path.join(kilosort_folder, 'templates.npy')
    chan_pos_path = os.path.join(kilosort_folder, 'channel_positions.npy')
    
    templates = np.load(temp_path)
    channel_depths = np.load(chan_pos_path)[:, 1]
    
    ptp = np.ptp(templates, axis=1)
    peak_channels = np.argmax(ptp, axis=1)
    
    for i, unit_id in enumerate(final_sort.unit_ids):
        if i < len(peak_channels):
            unit_depth_map[unit_id] = channel_depths[peak_channels[i]]

# 2. Prepare the Data for the Analyzer
# The analyzer expects an array of shape (num_units, 2) -> (x, y)
current_unit_ids = analyzer.sorting.unit_ids
new_locations = np.zeros((len(current_unit_ids), 2))

# We set X to 0 (center of probe) and Y to the correct depth
for i, unit_id in enumerate(current_unit_ids):
    if unit_id in unit_depth_map:
        depth = unit_depth_map[unit_id]
        new_locations[i, 1] = depth # Set Y (Depth)
        # Optional: Add random jitter to X so they don't overlap perfectly
        new_locations[i, 0] = np.random.uniform(0, 40) 
    else:
        new_locations[i, :] = np.nan # Mark missing ones

# 3. OVERWRITE the Analyzer's internal locations
# This bypasses the broken calculation and forces the plot to be correct
print("Injecting correct locations into Analyzer...")
analyzer.extensions['unit_locations'].data = new_locations

# 4. Plot
# Now the standard SI plotting function will see the correct data
plt.figure(figsize=(12, 8))
sw.plot_unit_locations(analyzer, backend="matplotlib")
plt.title("Analyzer Drift Map (Fixed via Injection)")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# 1. Re-Calculate the Correct Depths (The "Manual" method that worked)
print("Retrieving correct depths from Kilosort metadata...")
unit_depth_map = {}

# Try cluster_info first
info_path = os.path.join(kilosort_folder, 'cluster_info.tsv')
if os.path.exists(info_path):
    import pandas as pd
    df = pd.read_csv(info_path, sep='\t')
    for _, row in df.iterrows():
        unit_depth_map[row['cluster_id']] = row['depth']
else:
    # Fallback to templates
    print("Estimating from templates...")
    chan_pos_path = os.path.join(kilosort_folder, 'channel_positions.npy')
    temp_path = os.path.join(kilosort_folder, 'templates.npy')
    
    channel_depths = np.load(chan_pos_path)[:, 1]
    templates = np.load(temp_path)
    peak_channels = np.argmax(np.ptp(templates, axis=1), axis=1)
    
    for i, unit_id in enumerate(final_sort.unit_ids):
        if i < len(peak_channels):
            unit_depth_map[unit_id] = channel_depths[peak_channels[i]]

# 2. Get Spike Times from the Sliced Sorting (The specific session)
# We use the sorting attached to the analyzer to ensure we match the time window
sorting = analyzer.sorting
all_spikes = sorting.to_spike_vector()
spike_times = all_spikes['sample_index']
spike_unit_indices = all_spikes['unit_index']
unit_ids = sorting.unit_ids

# 3. Map Spikes to Depths
# Create a lookup array: Unit Index -> Depth
# (This is much faster than a loop for millions of spikes)
max_idx = np.max(spike_unit_indices)
index_to_depth = np.full(max_idx + 1, np.nan)

for i, unit_id in enumerate(unit_ids):
    if unit_id in unit_depth_map:
        # Map the specific unit index used in this slice to its depth
        index_to_depth[i] = unit_depth_map[unit_id]

# Apply mapping
spike_depths = index_to_depth[spike_unit_indices]

# Filter NaNs
valid_mask = ~np.isnan(spike_depths)
plot_times = spike_times[valid_mask]
plot_depths = spike_depths[valid_mask]

# Convert frames to seconds (optional, for X-axis)
fs = analyzer.recording.get_sampling_frequency()
plot_times_sec = plot_times / fs

# 4. Generate the Plot
print(f"Plotting {len(plot_times)} spikes...")
plt.figure(figsize=(12, 8))

# s=1 (small dots), alpha=0.1 (transparency)
plt.scatter(plot_times_sec, plot_depths, s=1, c='black', alpha=0.1, rasterized=True)

plt.title("Drift Map (Corrected Depths on Sliced Data)")
plt.xlabel("Time (seconds)")
plt.ylabel("Depth (um)")
plt.margins(x=0)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import spikeinterface.widgets as sw
import os

# 1. RUN THE ANALYZER (Standard Setup)
# We run the compute step to initialize the structure (even if the values are wrong/clumped)
print("Initializing Analyzer and computing defaults...")
analyzer.compute("unit_locations", method="center_of_mass")

# 2. GET THE "TRUTH" (Correct Depths from Kilosort Metadata)
# This mimics the manual plot logic that we know works
print("Retrieving correct depths from Kilosort metadata...")
unit_depth_map = {}

# Try cluster_info.tsv (Best source)
info_path = os.path.join(kilosort_folder, 'cluster_info.tsv')
if os.path.exists(info_path):
    import pandas as pd
    df = pd.read_csv(info_path, sep='\t')
    for _, row in df.iterrows():
        unit_depth_map[row['cluster_id']] = row['depth']
else:
    # Fallback to templates/channel_positions
    print("Estimating from templates...")
    chan_pos_path = os.path.join(kilosort_folder, 'channel_positions.npy')
    temp_path = os.path.join(kilosort_folder, 'templates.npy')
    
    channel_depths = np.load(chan_pos_path)[:, 1]
    templates = np.load(temp_path)
    # Peak channel index for every unit
    peak_channels = np.argmax(np.ptp(templates, axis=1), axis=1)
    
    for i, unit_id in enumerate(final_sort.unit_ids):
        if i < len(peak_channels):
            unit_depth_map[unit_id] = channel_depths[peak_channels[i]]

# 3. PREPARE THE DATA ARRAY
# We create an array (num_units, 2) to hold [x, y]
current_unit_ids = analyzer.sorting.unit_ids
new_locations = np.zeros((len(current_unit_ids), 2))

for i, unit_id in enumerate(current_unit_ids):
    if unit_id in unit_depth_map:
        # Set Y to the correct Depth
        new_locations[i, 1] = unit_depth_map[unit_id]
        
        # Set X to a random jitter (0-40um) so they don't overlap visually
        # (Or you can use the real X if available in cluster_info['group'])
        new_locations[i, 0] = np.random.uniform(0, 40)
    else:
        new_locations[i, :] = np.nan # Handle missing units

# 4. INJECT THE DATA (The Critical Fix)
# We update the 'unit_locations' KEY inside the data dictionary.
# Previous error was caused by overwriting .data directly.
extension = analyzer.get_extension("unit_locations")
extension.data['unit_locations'] = new_locations

print("Success: Correct locations injected into SpikeInterface Analyzer.")

# 5. VERIFY WITH SPIKEINTERFACE PLOT
# Now standard SI widgets should work perfectly
plt.figure(figsize=(12, 8))
sw.plot_unit_locations(analyzer, backend="matplotlib")
plt.title("Fixed SpikeInterface Drift Map")
plt.show()

In [ ]:
# we need to compute some required extensions
analyzer.compute(['random_spikes', 'templates', 'spike_amplitudes', 'spike_locations', 'noise_levels', 'quality_metrics'])
# note that spike_locations are optional, but recommended to compute accurate spike depths

# optionally, we can pass an LFP recording to compute RMS/PSD in the LFP band
recording_lfp = spre.bandpass_filter(final_rec, freq_min=1, freq_max=300)
# we can also decimate the LFP to speed up the process
recording_lfp = spre.decimate(recording_lfp, 10)

sexp.export_to_ibl_gui(
    sorting_analyzer=analyzer,
    output_folder=r'Z:\Saij\ephys\Mouse 2',
    lfp_recording=recording_lfp,
    n_jobs=-1
)

print("done")




In [ ]:
import spikeinterface.extractors as se
import spikeinterface as si
import numpy as np
import os
import matplotlib.pyplot as plt

# --- 1. SETUP ---
raw_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28\Record Node 101"
ks_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

# --- 2. DEFINE EXPLICIT SLICE WINDOWS ---
# We verify the triggers are scalar values (integers)
# We assume triggers_flat is [start, ..., end] for LOCAL time
# We assume triggers_post is [start, ..., end] for GLOBAL time

margin = 100

# RECORDING uses LOCAL triggers (triggers_flat)
rec_start = int(triggers_flat[0]) - margin
rec_end   = int(triggers_flat[-1]) + margin

# SORTING uses GLOBAL triggers (triggers_post)
sort_start = int(triggers_post[0]) - margin
sort_end   = int(triggers_post[-1]) + margin

print(f"Slicing Recording (Local): {rec_start} to {rec_end}")
print(f"Slicing Sorting   (Global): {sort_start} to {sort_end}")


# --- 3. PREPARE RECORDING (The "Kilosort-Compatible" Fix) ---

# A. Load Raw
rec_raw = se.read_openephys(raw_path)

# B. Check bounds before slicing!
total_frames = rec_raw.get_num_frames()
if rec_end > total_frames:
    print(f"WARNING: rec_end ({rec_end}) > total_frames ({total_frames}). Clamping to max.")
    rec_end = total_frames

# C. Rename & Map Channels
# This fixes the "Clumping" / "Channel 0" bug
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
rec_aligned = rec_raw.select_channels(channel_map)

# D. Apply Slice
start_safe = max(0, rec_start)
rec_final = rec_aligned.frame_slice(start_frame=start_safe, end_frame=rec_end)
print(f"Synced Recording Duration: {rec_final.get_num_frames()} frames")


# --- 4. PREPARE SORTING ---

# Slice Global Sorting using Global Triggers
sort_start_safe = max(0, sort_start)
sort_final = final_sort.frame_slice(start_frame=sort_start_safe, end_frame=sort_end)
print(f"Synced Sorting Units: {len(sort_final.get_unit_ids())}")
print(f"Synced Spikes: {sort_final.get_total_num_spikes()}")


# --- 5. RUN ANALYZER ---

if sort_final.get_total_num_spikes() == 0:
    print("CRITICAL ERROR: Sliced sorting is EMPTY. Check if 'sort_start' / 'sort_end' cover the correct range.")
else:
    print("Running Analyzer...")
    analyzer = si.create_sorting_analyzer(
        sorting=sort_final,
        recording=rec_final,
        format="memory",
        sparse=True
    )
    
    # Compute everything (Now works automatically because channels match)
    analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=50)
    analyzer.compute("waveforms")
    analyzer.compute("templates")
    analyzer.compute("unit_locations", method="center_of_mass")
    
    # Plot
    import spikeinterface.widgets as sw
    plt.figure(figsize=(12, 12))
    sw.plot_unit_locations(analyzer, backend="matplotlib")
    plt.title("Final Corrected Drift Map")
    plt.show()

In [ ]:
import spikeinterface.extractors as se
import spikeinterface as si
import numpy as np
import os
import matplotlib.pyplot as plt

# --- 1. SETUP PATHS ---
raw_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28\Record Node 101"
ks_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

# --- 2. DEFINE SLICE WINDOWS (Correct Mapping) ---

margin = 100

# RECORDING is the Raw Segment (starts at 0) -> Uses triggers_flat (Local)
rec_start = int(triggers_post[0]) - margin
rec_end   = int(triggers_post[-1]) + margin

# SORTING is Concatenated (Global Time) -> Uses triggers_post (Global)
sort_start = int(triggers_flat[0]) - margin
sort_end   = int(triggers_flat[-1]) + margin

print("--- Alignment Check ---")
print(f"Recording Slice (Local):  {rec_start} to {rec_end}")
print(f"Sorting Slice   (Global): {sort_start} to {sort_end}")


# --- 3. PREPARE RECORDING (Fix Channels + Slice) ---

# A. Load Raw
rec_raw = se.read_openephys(raw_path)

# B. Sanity Check
# Ensure we don't slice past the end of the file
total_frames = rec_raw.get_num_frames()
if rec_end > total_frames:
    print(f"WARNING: rec_end ({rec_end}) exceeds file length ({total_frames}). Clamping.")
    rec_end = total_frames

# C. Fix Channel Map (The Permanent Fix)
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
rec_aligned = rec_raw.select_channels(channel_map)

# D. Apply Slice (Using Local Triggers)
start_safe_rec = max(0, rec_start)
rec_final = rec_aligned.frame_slice(start_frame=start_safe_rec, end_frame=rec_end)

print(f"Synced Recording Duration: {rec_final.get_num_frames()} frames")


# --- 4. PREPARE SORTING (Slice Global) ---

# Apply Slice (Using Global Triggers)
# This keeps the spikes in the window AND subtracts the offset so t=0 matches the recording
start_safe_sort = max(0, sort_start)
sort_final = final_sort.frame_slice(start_frame=start_safe_sort, end_frame=sort_end)

print(f"Synced Sorting Units: {len(sort_final.get_unit_ids())}")
print(f"Synced Spikes: {sort_final.get_total_num_spikes()}")


# --- 5. VERIFY ALIGNMENT (Blob Check) ---

if sort_final.get_total_num_spikes() > 0:
    # Check first unit with spikes
    unit_ids = sort_final.get_unit_ids()
    first_spike = None
    for uid in unit_ids:
        spikes = sort_final.get_unit_spike_train(uid)
        if len(spikes) > 0:
            first_spike = spikes[0]
            break
            
    if first_spike is not None:
        # Extract raw data at the spike time
        # Since both are now sliced to the same window, t=first_spike in sorting 
        # should match t=first_spike in recording
        window = 100
        start = max(0, first_spike - window)
        end = first_spike + window
        
        snippet = rec_final.get_traces(start_frame=start, end_frame=end)
        
        plt.figure(figsize=(6, 6))
        plt.imshow(snippet.T, aspect='auto', cmap='RdBu', vmin=-100, vmax=100, origin='lower')
        plt.axvline(x=window, color='green', linestyle='--', label='Spike Time')
        plt.title("Alignment Verification\n(Look for blob at green line)")
        plt.legend()
        plt.show()

# --- 6. RUN ANALYZER ---

if sort_final.get_total_num_spikes() > 0:
    print("Running Analyzer...")
    analyzer = si.create_sorting_analyzer(
        sorting=sort_final,
        recording=rec_final,
        format="memory",
        sparse=True
    )
    
    # Compute using the now-perfectly aligned data
    analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=50)
    analyzer.compute("waveforms")
    analyzer.compute("templates")
    analyzer.compute("unit_locations", method="center_of_mass")
    
    # Final Plot
    plt.figure(figsize=(10, 10))
    sw.plot_unit_locations(analyzer, backend="matplotlib")
    plt.title("Final Corrected Drift Map")
    plt.show()
else:
    print("CRITICAL: Sliced sorting is empty. Verify that triggers_post overlaps with the spike times.")

In [ ]:
rec_lfp = spre.bandpass_filter(rec_raw, freq_min=1, freq_max=300)

In [ ]:
s = sw.plot_traces(recording=rec_lfp, order_channel_by_depth = True, backend="matplotlib")

In [ ]:
import spikeinterface.extractors as se
import numpy as np
import matplotlib.pyplot as plt
import os

# --- 1. SETTINGS ---
# Paths
raw_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28\Record Node 101"
ks_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

# LFP Window (seconds)
pre_time = 0.5
post_time = 1.0
downsample_factor = 30 # Plotting 30kHz is heavy; 30x downsample -> 1kHz

# --- 2. LOAD & FIX RECORDING ---
# We apply the channel map fix so we look at the correct physical channels
rec_raw = se.read_openephys(raw_path)
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
rec = rec_raw.select_channels(channel_map)

fs = rec.get_sampling_frequency()
print(f"Loaded Recording: {fs} Hz, {rec.get_num_channels()} channels")

# --- 3. PREPARE TRIGGERS (Offset Correction) ---
triggers = np.array(triggers_post, dtype=int)

# Check if triggers are "Global" (huge numbers) vs "Local" (fit in file)
if triggers[0] > rec.get_num_frames():
    print(f"Detected Global Triggers (Start: {triggers[0]}).")
    print(f"Recording Length: {rec.get_num_frames()}.")
    print("Auto-correcting: Subtracting offset so first trigger aligns with start of interest.")
    
    # Assuming the first trigger in 'triggers_post' corresponds to the 
    # start of the relevant events in this local file.
    # Note: If 'triggers_flat' is available, calculating offset = post[0] - flat[0] is safer.
    # Here we assume we normalize to the first trigger for visualization.
    offset = triggers[0] - 1000 # preserving relative timing, starting 1000 frames in
    
    # If you have the exact offset from previous steps, uncomment and set it here:
    # offset = 90000000 
    
    local_triggers = triggers - offset
else:
    print("Triggers appear to be Local. Using as-is.")
    local_triggers = triggers[1:]

# Filter triggers that are within the file bounds
valid_mask = (local_triggers > (pre_time * fs)) & (local_triggers < (rec.get_num_frames() - post_time * fs))
local_triggers = local_triggers[valid_mask]
print(f"Plotting {len(local_triggers)} valid triggers.")


# --- 4. EXTRACT LFP ---
num_samples = int((pre_time + post_time) * fs)
num_downsampled = num_samples // downsample_factor

# Array to store sum of responses: (Channels, Time)
lfp_sum = np.zeros((rec.get_num_channels(), num_downsampled))
count = 0

print("Extracting and averaging traces...")

for trig in local_triggers:
    start_frame = int(trig - pre_time * fs)
    end_frame = int(trig + post_time * fs)
    
    # Extract
    trace = rec.get_traces(start_frame=start_frame, end_frame=end_frame)
    
    # Transpose to (Channels, Time)
    trace = trace.T 
    
    # Simple downsampling (take every Nth sample)
    trace_ds = trace[:, ::downsample_factor]
    
    # Handle edge cases where downsampling rounding varies slightly
    if trace_ds.shape[1] == num_downsampled:
        lfp_sum += trace_ds
        count += 1

# Compute Average
lfp_avg = lfp_sum / count

# --- 5. PLOT ---
time_axis = np.linspace(-pre_time, post_time, num_downsampled)

plt.figure(figsize=(12, 10))

# Plot 1: Heatmap (All Channels)
plt.subplot(2, 1, 1)
# vmin/vmax set contrast. Adjust based on your signal magnitude (e.g. +/- 100 uV)
plt.imshow(lfp_avg, aspect='auto', extent=[-pre_time, post_time, rec.get_num_channels(), 0], 
           cmap='RdBu_r', vmin=-100, vmax=100)
plt.colorbar(label="Voltage (uV)")
plt.axvline(0, color='k', linestyle='--', linewidth=1)
plt.title(f"Average LFP Response (Heatmap) - {count} Trials")
plt.ylabel("Channel Index (Top to Bottom)")
plt.xlabel("Time from Trigger (s)")

# Plot 2: Butterfly Plot (All traces overlaid)
plt.subplot(2, 1, 2)
plt.plot(time_axis, lfp_avg.T, color='k', alpha=0.1, linewidth=0.5)
# Plot the mean of all channels in Red
plt.plot(time_axis, np.mean(lfp_avg, axis=0), color='r', linewidth=2, label='Grand Mean')
plt.axvline(0, color='b', linestyle='--')
plt.title("LFP Traces (All Channels)")
plt.xlabel("Time from Trigger (s)")
plt.ylabel("Voltage (uV)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import spikeinterface.extractors as se
import numpy as np
import matplotlib.pyplot as plt
import os

# --- 1. SETUP ---
raw_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28\Record Node 101"
ks_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

# Parameters
pre_time = 0.5
post_time = 1.0
downsample_factor = 30 # 30kHz -> 1kHz
vmin, vmax = -100, 100 # Contrast limits (uV)

# --- 2. LOAD DATA ---
rec_raw = se.read_openephys(raw_path)
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
rec = rec_raw.select_channels(channel_map)
fs = rec.get_sampling_frequency()

# --- 3. SORT CHANNELS BY DEPTH ---
# Load channel positions (N_channels, 2) -> (x, y)
chan_pos = np.load(os.path.join(ks_path, 'channel_positions.npy'))
depths = chan_pos[:, 1] # Y-coordinate

# Get indices that sort the depths from Low to High
sorted_indices = np.argsort(depths)
sorted_depths = depths[sorted_indices]

print(f"Sorting {len(depths)} channels by depth ({min(depths)}-{max(depths)} um)")

# --- 4. PREPARE TRIGGERS ---
# Use indices 1 to 150
subset_triggers = np.array(triggers_post[1:151], dtype=int)

# Auto-correct Global Offset if needed
if subset_triggers[0] > rec.get_num_frames():
    # If triggers are huge (Global), subtract the file offset
    # Assuming 'triggers_post[0]' aligns with 'triggers_flat[0]' logic
    offset = subset_triggers[0] - (1.0 * fs) # Rough alignment to start of file if flat triggers unknown
    local_triggers = subset_triggers - offset
    print(f"Corrected global triggers with offset: {offset}")
else:
    local_triggers = subset_triggers

# Filter valid bounds
valid_mask = (local_triggers > (pre_time * fs)) & (local_triggers < (rec.get_num_frames() - post_time * fs))
final_triggers = local_triggers[valid_mask]
print(f"Averaging {len(final_triggers)} trials...")

# --- 5. EXTRACT & AVERAGE ---
num_samples = int((pre_time + post_time) * fs)
num_ds = num_samples // downsample_factor

# Shape: (Channels, Time)
lfp_sum = np.zeros((rec.get_num_channels(), num_ds))
count = 0

for trig in final_triggers:
    start = int(trig - pre_time * fs)
    end = int(trig + post_time * fs)
    
    # Extract
    trace = rec.get_traces(start_frame=start, end_frame=end).T
    # Downsample
    trace_ds = trace[:, ::downsample_factor]
    
    if trace_ds.shape[1] == num_ds:
        lfp_sum += trace_ds
        count += 1

# Calculate Mean
lfp_avg = lfp_sum / count

# Sort the rows by depth
lfp_sorted = lfp_avg[sorted_indices, :]

# --- 6. PLOT ---
plt.figure(figsize=(10, 8))

# Extent defines the axes: [x_min, x_max, y_min, y_max]
# We use sorted_depths[0] and [-1] for the Y-axis range
extent = [-pre_time, post_time, sorted_depths[0], sorted_depths[-1]]

im = plt.imshow(lfp_sorted, aspect='auto', origin='lower', extent=extent,
           cmap='RdBu_r', vmin=vmin, vmax=vmax)

plt.colorbar(im, label="Voltage (uV)")
plt.axvline(0, color='k', linestyle='--', linewidth=1, alpha=0.7)

plt.title(f"Average LFP by Depth (Triggers 1-150)\nN={count} trials")
plt.xlabel("Time from Trigger (s)")
plt.ylabel("Depth (microns)")

plt.tight_layout()
plt.show()

In [ ]:
import spikeinterface.extractors as se
import numpy as np
import matplotlib.pyplot as plt
import os
from ipywidgets import interact, IntSlider
import ipywidgets as widgets

# --- 1. SETUP ---
raw_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28\Record Node 101"
ks_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

# Parameters
pre_time = 0.1
post_time = 0.5
downsample_factor = 30 

# --- 2. LOAD DATA ---
rec_raw = se.read_openephys(raw_path)
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
rec = rec_raw.select_channels(channel_map)
fs = rec.get_sampling_frequency()

# --- 3. SORT CHANNELS BY DEPTH ---
chan_pos = np.load(os.path.join(ks_path, 'channel_positions.npy'))
depths = chan_pos[:, 1]
original_ids = rec.channel_ids

# Sort everything by depth (Low -> High)
sorted_indices = np.argsort(depths)
sorted_depths = depths[sorted_indices]
sorted_ids = original_ids[sorted_indices]

print(f"Data loaded: {len(depths)} channels sorted by depth.")

# --- 4. PROCESS TRIGGERS (1-150) ---
subset_triggers = np.array(triggers_post[1:151], dtype=int)

# Auto-correct Global Offset
if subset_triggers[0] > rec.get_num_frames():
    offset = subset_triggers[0] - (1.0 * fs) 
    local_triggers = subset_triggers - offset
    print(f"Applied Global Offset: {offset}")
else:
    local_triggers = subset_triggers

# Filter bounds
valid_mask = (local_triggers > (pre_time * fs)) & (local_triggers < (rec.get_num_frames() - post_time * fs))
final_triggers = local_triggers[valid_mask]

# --- 5. COMPUTE AVERAGE LFP ---
print(f"Averaging {len(final_triggers)} trials (this may take a moment)...")

num_samples = int((pre_time + post_time) * fs)
num_ds = num_samples // downsample_factor

lfp_sum = np.zeros((rec.get_num_channels(), num_ds))
count = 0

for trig in final_triggers:
    start = int(trig - pre_time * fs)
    end = int(trig + post_time * fs)
    trace = rec.get_traces(start_frame=start, end_frame=end).T
    lfp_sum += trace[:, ::downsample_factor]
    count += 1

# Average and Sort
lfp_avg = lfp_sum / count
lfp_sorted = lfp_avg[sorted_indices, :]
time_axis = np.linspace(-pre_time, post_time, num_ds)

print("Calculation Complete. Generating Widget...")

# --- 6. INTERACTIVE PLOTTER ---

def plot_channel(index):
    """
    Plots a single channel based on its sorted index (0 = Deepest).
    """
    # Get data for this sorted index
    trace = lfp_sorted[index]
    depth = sorted_depths[index]
    orig_id = sorted_ids[index]
    
    plt.figure(figsize=(10, 5))
    
    # Plot the Average Trace
    plt.plot(time_axis, trace, color='black', linewidth=1.5)
    
    # Add T=0 marker
    plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='Trigger')
    plt.axhline(0, color='gray', linewidth=0.5)
    
    # Styling
    plt.title(f"Channel Index: {index} (Orig ID: {orig_id})\nDepth: {depth} $\mu m$", fontsize=14)
    plt.xlabel("Time from Trigger (s)")
    plt.ylabel("Voltage ($\mu V$)")
    plt.xlim(-pre_time, post_time)
    
    # Dynamic Y-Limits based on specific channel data (zooms in on small signals)
    # Or set fixed limits like plt.ylim(-100, 100) to compare amplitude
    buffer = (np.max(trace) - np.min(trace)) * 0.1
    plt.ylim(np.min(trace) - buffer, np.max(trace) + buffer)
    
    plt.grid(True, alpha=0.3)
    plt.show()

# Create the Slider Widget
interact(
    plot_channel, 
    index=IntSlider(
        min=0, 
        max=len(sorted_indices)-1, 
        step=1, 
        value=0, 
        description='Sort Index:',
        continuous_update=False # Updates only when you release mouse (smoother)
    )
);

In [ ]:
import spikeinterface.extractors as se
import numpy as np
import matplotlib.pyplot as plt
import os
from ipywidgets import interact, IntSlider
import ipywidgets as widgets

# --- 1. SETUP ---
raw_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28\Record Node 101"
ks_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

# Parameters
pre_time = 0.5
post_time = 1.0
downsample_factor = 30 

# --- 2. LOAD DATA ---
rec_raw = se.read_openephys(raw_path)
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
rec = rec_raw.select_channels(channel_map)
fs = rec.get_sampling_frequency()

# --- 3. SORT CHANNELS BY DEPTH ---
chan_pos = np.load(os.path.join(ks_path, 'channel_positions.npy'))
depths = chan_pos[:, 1]
original_ids = rec.channel_ids

# Sort everything by depth (Low -> High)
sorted_indices = np.argsort(depths)
sorted_depths = depths[sorted_indices]
sorted_ids = original_ids[sorted_indices]

print(f"Data loaded: {len(depths)} channels sorted by depth.")

# --- 4. PROCESS TRIGGERS (1-150) ---
subset_triggers = np.array(triggers_post[1:151], dtype=int)

if subset_triggers[0] > rec.get_num_frames():
    offset = subset_triggers[0] - (1.0 * fs) 
    local_triggers = subset_triggers - offset
else:
    local_triggers = subset_triggers

valid_mask = (local_triggers > (pre_time * fs)) & (local_triggers < (rec.get_num_frames() - post_time * fs))
final_triggers = local_triggers[valid_mask]

# --- 5. COMPUTE AVERAGE LFP ---
print(f"Averaging {len(final_triggers)} trials...")

num_samples = int((pre_time + post_time) * fs)
num_ds = num_samples // downsample_factor

lfp_sum = np.zeros((rec.get_num_channels(), num_ds))
count = 0

for trig in final_triggers:
    start = int(trig - pre_time * fs)
    end = int(trig + post_time * fs)
    trace = rec.get_traces(start_frame=start, end_frame=end).T
    lfp_sum += trace[:, ::downsample_factor]
    count += 1

# Average
lfp_avg = lfp_sum / count

# Sort by depth
lfp_sorted = lfp_avg[sorted_indices, :]
time_axis = np.linspace(-pre_time, post_time, num_ds)

# --- NEW STEP: NORMALIZE AT TRIGGER ONSET ---
# Find the index closest to t=0
zero_idx = np.abs(time_axis - 0).argmin()

# Subtract the value at t=0 from the whole trace for each channel
# shape: (N_channels, 1) broadcasted to (N_channels, N_timepoints)
baseline_values = lfp_sorted[:, zero_idx][:, np.newaxis] 
lfp_normalized = lfp_sorted - baseline_values

print("Normalization complete. Generating Widget...")

# --- 6. INTERACTIVE PLOTTER ---

def plot_channel(index):
    """
    Plots a single channel based on its sorted index (0 = Deepest).
    """
    # Use normalized data
    trace = lfp_normalized[index]
    depth = sorted_depths[index]
    orig_id = sorted_ids[index]
    
    plt.figure(figsize=(10, 5))
    
    # Plot the Average Trace
    plt.plot(time_axis, trace, color='black', linewidth=1.5)
    
    # Add markers
    plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='Trigger Onset')
    plt.axhline(0, color='blue', linewidth=1, linestyle='-', alpha=0.3, label='Baseline (0 uV)')
    
    # Styling
    plt.title(f"Channel Index: {index} (Orig ID: {orig_id})\nDepth: {depth} $\mu m$ (Normalized)", fontsize=14)
    plt.xlabel("Time from Trigger (s)")
    plt.ylabel("Voltage ($\mu V$) - Relative to Onset")
    plt.xlim(-pre_time, post_time)
    
    # Dynamic Y-Limits
    buffer = (np.max(trace) - np.min(trace)) * 0.1
    if buffer == 0: buffer = 10 # Safety for flat lines
    plt.ylim(np.min(trace) - buffer, np.max(trace) + buffer)
    
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.show()

interact(
    plot_channel, 
    index=IntSlider(
        min=0, 
        max=len(sorted_indices)-1, 
        step=1, 
        value=0, 
        description='Sort Index:',
        continuous_update=False 
    )
);

In [ ]:
import spikeinterface.extractors as se
import numpy as np
import matplotlib.pyplot as plt
import os
from ipywidgets import interact, IntSlider
import ipywidgets as widgets

# --- 1. SETUP ---
raw_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Raw Data\mouse2_renewal_checkerboard_2025-06-12_12-52-28\Record Node 101"
ks_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 2\Renewal\Neural Data\Concatenated Data\kilosort4" 

# Parameters
pre_time = 0.5
post_time = 1.0
downsample_factor = 30 

# --- 2. LOAD DATA ---
rec_raw = se.read_openephys(raw_path)
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
rec = rec_raw.select_channels(channel_map)
fs = rec.get_sampling_frequency()

# --- 3. SORT CHANNELS BY DEPTH ---
chan_pos = np.load(os.path.join(ks_path, 'channel_positions.npy'))
depths = chan_pos[:, 1]
original_ids = rec.channel_ids

# Sort everything by depth (Low -> High)
sorted_indices = np.argsort(depths)
sorted_depths = depths[sorted_indices]
sorted_ids = original_ids[sorted_indices]

print(f"Data loaded: {len(depths)} channels sorted by depth.")

# --- 4. PROCESS TRIGGERS (1-150) ---
subset_triggers = np.array(triggers_post[1:151], dtype=int)

if subset_triggers[0] > rec.get_num_frames():
    offset = subset_triggers[0] - (1.0 * fs) 
    local_triggers = subset_triggers - offset
else:
    local_triggers = subset_triggers

valid_mask = (local_triggers > (pre_time * fs)) & (local_triggers < (rec.get_num_frames() - post_time * fs))
final_triggers = local_triggers[valid_mask]

# --- 5. COMPUTE AVERAGE LFP ---
print(f"Averaging {len(final_triggers)} trials...")

num_samples = int((pre_time + post_time) * fs)
num_ds = num_samples // downsample_factor

lfp_sum = np.zeros((rec.get_num_channels(), num_ds))
count = 0

for trig in final_triggers:
    start = int(trig - pre_time * fs)
    end = int(trig + post_time * fs)
    trace = rec.get_traces(start_frame=start, end_frame=end).T
    lfp_sum += trace[:, ::downsample_factor]
    count += 1

# Average
lfp_avg = lfp_sum / count

# Sort by depth
lfp_sorted = lfp_avg[sorted_indices, :]
time_axis = np.linspace(-pre_time, post_time, num_ds)

# --- NEW STEP: NORMALIZE AT TRIGGER ONSET ---
# Find the index closest to t=0
zero_idx = np.abs(time_axis - 0).argmin()

# Subtract the value at t=0 from the whole trace for each channel
# shape: (N_channels, 1) broadcasted to (N_channels, N_timepoints)
baseline_values = lfp_sorted[:, zero_idx][:, np.newaxis] 
lfp_normalized = lfp_sorted - baseline_values

print("Normalization complete. Generating Widget...")

# --- 6. INTERACTIVE PLOTTER ---

def plot_channel(index):
    """
    Plots a single channel based on its sorted index (0 = Deepest).
    """
    # Use normalized data
    trace = lfp_normalized[index]
    depth = sorted_depths[index]
    orig_id = sorted_ids[index]
    
    plt.figure(figsize=(10, 5))
    
    # Plot the Average Trace
    plt.plot(time_axis, trace, color='black', linewidth=1.5)
    
    # Add markers
    plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='Trigger Onset')
    plt.axhline(0, color='blue', linewidth=1, linestyle='-', alpha=0.3, label='Baseline (0 uV)')
    
    # Styling
    plt.title(f"Channel Index: {index} (Orig ID: {orig_id})\nDepth: {depth} $\mu m$ (Normalized)", fontsize=14)
    plt.xlabel("Time from Trigger (s)")
    plt.ylabel("Voltage ($\mu V$) - Relative to Onset")
    plt.xlim(-pre_time, post_time)
    
    # Dynamic Y-Limits
    buffer = (np.max(trace) - np.min(trace)) * 0.1
    if buffer == 0: buffer = 10 # Safety for flat lines
    plt.ylim(np.min(trace) - buffer, np.max(trace) + buffer)
    
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.show()

interact(
    plot_channel, 
    index=IntSlider(
        min=0, 
        max=len(sorted_indices)-1, 
        step=1, 
        value=0, 
        description='Sort Index:',
        continuous_update=False 
    )
);

In [ ]:
import spikeinterface.extractors as se
import numpy as np
import matplotlib.pyplot as plt
import os
from ipywidgets import interact, IntSlider
import ipywidgets as widgets

# --- 1. SETUP ---
raw_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 3\Renewal\Neural Data\Raw Data\mouse3_renewal_checkerboard"


# Parameters
pre_time = 0.1
post_time = 0.5
downsample_factor = 30 

# --- 2. LOAD DATA ---
rec_raw = se.read_openephys(raw_path)
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
rec = rec_raw.select_channels(channel_map)
fs = rec.get_sampling_frequency()

# --- 3. SORT CHANNELS BY DEPTH ---
chan_pos = np.load(os.path.join(ks_path, 'channel_positions.npy'))
depths = chan_pos[:, 1]
original_ids = rec.channel_ids

# Sort everything by depth (Low -> High)
sorted_indices = np.argsort(depths)
sorted_depths = depths[sorted_indices]
sorted_ids = original_ids[sorted_indices]

print(f"Data loaded: {len(depths)} channels sorted by depth.")

# --- 4. PROCESS TRIGGERS (1-150) ---
subset_triggers = np.array(triggers_post[1:151], dtype=int)

if subset_triggers[0] > rec.get_num_frames():
    offset = subset_triggers[0] - (1.0 * fs) 
    local_triggers = subset_triggers - offset
else:
    local_triggers = subset_triggers

valid_mask = (local_triggers > (pre_time * fs)) & (local_triggers < (rec.get_num_frames() - post_time * fs))
final_triggers = local_triggers[valid_mask]

# --- 5. COMPUTE AVERAGE LFP ---
print(f"Averaging {len(final_triggers)} trials...")

num_samples = int((pre_time + post_time) * fs)
num_ds = num_samples // downsample_factor

lfp_sum = np.zeros((rec.get_num_channels(), num_ds))
count = 0

for trig in final_triggers:
    start = int(trig - pre_time * fs)
    end = int(trig + post_time * fs)
    trace = rec.get_traces(start_frame=start, end_frame=end).T
    lfp_sum += trace[:, ::downsample_factor]
    count += 1

# Average
lfp_avg = lfp_sum / count

# Sort by depth
lfp_sorted = lfp_avg[sorted_indices, :]
time_axis = np.linspace(-pre_time, post_time, num_ds)

# --- NEW STEP: NORMALIZE AT TRIGGER ONSET ---
# Find the index closest to t=0
zero_idx = np.abs(time_axis - 0).argmin()

# Subtract the value at t=0 from the whole trace for each channel
# shape: (N_channels, 1) broadcasted to (N_channels, N_timepoints)
baseline_values = lfp_sorted[:, zero_idx][:, np.newaxis] 
lfp_normalized = lfp_sorted - baseline_values

print("Normalization complete. Generating Widget...")

# --- 6. INTERACTIVE PLOTTER ---

def plot_channel(index):
    """
    Plots a single channel based on its sorted index (0 = Deepest).
    """
    # Use normalized data
    trace = lfp_normalized[index]
    depth = sorted_depths[index]
    orig_id = sorted_ids[index]
    
    plt.figure(figsize=(10, 5))
    
    # Plot the Average Trace
    plt.plot(time_axis, trace, color='black', linewidth=1.5)
    
    # Add markers
    plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='Trigger Onset')
    plt.axhline(0, color='blue', linewidth=1, linestyle='-', alpha=0.3, label='Baseline (0 uV)')
    
    # Styling
    plt.title(f"Channel Index: {index} (Orig ID: {orig_id})\nDepth: {depth} $\mu m$ (Normalized)", fontsize=14)
    plt.xlabel("Time from Trigger (s)")
    plt.ylabel("Voltage ($\mu V$) - Relative to Onset")
    plt.xlim(-pre_time, post_time)
    
    # Dynamic Y-Limits
    buffer = (np.max(trace) - np.min(trace)) * 0.1
    if buffer == 0: buffer = 10 # Safety for flat lines
    plt.ylim(np.min(trace) - buffer, np.max(trace) + buffer)
    
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.show()

interact(
    plot_channel, 
    index=IntSlider(
        min=0, 
        max=len(sorted_indices)-1, 
        step=1, 
        value=0, 
        description='Sort Index:',
        continuous_update=False 
    )
);

In [ ]:
import spikeinterface.extractors as se
import numpy as np
import matplotlib.pyplot as plt
import os
from ipywidgets import interact, IntSlider
import ipywidgets as widgets

# --- 1. SETUP ---
raw_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 3\Renewal\Neural Data\Raw Data\mouse3_renewal_checkerboard"
 

# Parameters
pre_time = 0.1
post_time = 0.5
downsample_factor = 30 

# --- 2. LOAD DATA ---
rec_raw = se.read_openephys(raw_path)
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
rec = rec_raw.select_channels(channel_map)
fs = rec.get_sampling_frequency()

# --- 3. SORT CHANNELS BY DEPTH ---
chan_pos = np.load(os.path.join(ks_path, 'channel_positions.npy'))
depths = chan_pos[:, 1]
original_ids = rec.channel_ids

sorted_indices = np.argsort(depths)
sorted_depths = depths[sorted_indices]
sorted_ids = original_ids[sorted_indices]

print(f"Data loaded: {len(depths)} channels sorted by depth.")

# --- 4. PROCESS TRIGGERS (1-150) ---
subset_triggers = np.array(triggers_post[1:151], dtype=int)

if subset_triggers[0] > rec.get_num_frames():
    offset = subset_triggers[0] - (1.0 * fs) 
    local_triggers = subset_triggers - offset
else:
    local_triggers = subset_triggers

valid_mask = (local_triggers > (pre_time * fs)) & (local_triggers < (rec.get_num_frames() - post_time * fs))
final_triggers = local_triggers[valid_mask]

# --- 5. EXTRACT ALL TRIALS ---
print(f"Extracting {len(final_triggers)} trials for SEM calculation...")

num_samples = int((pre_time + post_time) * fs)
num_ds = num_samples // downsample_factor

# List to store every single trial: [ (N_ch, N_time), (N_ch, N_time), ... ]
all_trials_list = []

for trig in final_triggers:
    start = int(trig - pre_time * fs)
    end = int(trig + post_time * fs)
    trace = rec.get_traces(start_frame=start, end_frame=end).T
    
    # Downsample
    trace_ds = trace[:, ::downsample_factor]
    
    # Ensure correct shape before appending
    if trace_ds.shape[1] == num_ds:
        all_trials_list.append(trace_ds)

# Convert to 3D Array: (N_trials, N_channels, N_timepoints)
all_trials = np.array(all_trials_list)

# --- 6. BASELINE CORRECTION & STATISTICS ---
time_axis = np.linspace(-pre_time, post_time, num_ds)
zero_idx = np.abs(time_axis - 0).argmin()

# Subtract the value at t=0 for EACH trial individually
# Value at zero: (N_trials, N_channels, 1)
baseline_vals = all_trials[:, :, zero_idx][:, :, np.newaxis]
all_trials_norm = all_trials - baseline_vals

# Calculate Mean and SEM across the "Trials" axis (axis 0)
mean_trace = np.mean(all_trials_norm, axis=0) # (N_channels, N_time)
std_trace = np.std(all_trials_norm, axis=0)
sem_trace = std_trace / np.sqrt(len(all_trials_list))

# Sort by Depth
mean_sorted = mean_trace[sorted_indices, :]
sem_sorted = sem_trace[sorted_indices, :]

print("Statistics complete. Generating Widget...")

# --- 7. INTERACTIVE PLOTTER ---

def plot_channel(index):
    """
    Plots Mean +/- SEM for a single channel.
    """
    # Get data
    mean = mean_sorted[index]
    sem = sem_sorted[index]
    depth = sorted_depths[index]
    orig_id = sorted_ids[index]
    
    plt.figure(figsize=(10, 5))
    
    # Plot Mean Line
    plt.plot(time_axis, mean, color='black', linewidth=1.5, label='Mean Response')
    
    # Plot SEM Shading
    plt.fill_between(time_axis, mean - sem, mean + sem, 
                     color='black', alpha=0.2, label='SEM')
    
    # Add Markers
    plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='Trigger Onset')
    plt.axhline(0, color='blue', linewidth=1, linestyle='-', alpha=0.2)
    
    # Styling
    plt.title(f"Channel Index: {index} (Orig ID: {orig_id})\nDepth: {depth} $\mu m$ (Normalized)", fontsize=14)
    plt.xlabel("Time from Trigger (s)")
    plt.ylabel("Voltage ($\mu V$)")
    plt.xlim(-pre_time, post_time)
    
    # Dynamic Y-Limits (include the SEM in the bounds calculation)
    upper_bound = np.max(mean + sem)
    lower_bound = np.min(mean - sem)
    buffer = (upper_bound - lower_bound) * 0.1
    if buffer == 0: buffer = 10
    
    plt.ylim(lower_bound - buffer, upper_bound + buffer)
    
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.show()

interact(
    plot_channel, 
    index=IntSlider(
        min=0, 
        max=len(sorted_indices)-1, 
        step=1, 
        value=0, 
        description='Sort Index:',
        continuous_update=False 
    )
);


In [ ]:
import spikeinterface.extractors as se


# Convert to 3D Array: (N_trials, N_channels, N_timepoints)
all_trials = np.array(all_trials_list)

# --- 6. BASELINE CORRECTION & STATISTICS ---
time_axis = np.linspace(-pre_time, post_time, num_ds)
zero_idx = np.abs(time_axis - 0).argmin()

# Subtract the value at t=0 for EACH trial individually
# Value at zero: (N_trials, N_channels, 1)
baseline_vals = all_trials[:, :, zero_idx][:, :, np.newaxis]
all_trials_norm = all_trials - baseline_vals

# Calculate Mean and SEM across the "Trials" axis (axis 0)
mean_trace = np.mean(all_trials_norm, axis=0) # (N_channels, N_time)
std_trace = np.std(all_trials_norm, axis=0)
sem_trace = std_trace / np.sqrt(len(all_trials_list))

# Sort by Depth
mean_sorted = mean_trace[sorted_indices, :]
sem_sorted = sem_trace[sorted_indices, :]

print("Statistics complete. Generating Widget...")

# --- 7. INTERACTIVE PLOTTER ---

def plot_channel(index):
    """
    Plots Mean +/- SEM for a single channel.
    """
    # Get data
    mean = mean_sorted[index]
    sem = sem_sorted[index]
    depth = sorted_depths[index]
    orig_id = sorted_ids[index]
    
    plt.figure(figsize=(10, 5))
    
    # Plot Mean Line
    plt.plot(time_axis, mean, color='black', linewidth=1.5, label='Mean Response')
    
    # Plot SEM Shading
    plt.fill_between(time_axis, mean - sem, mean + sem, 
                     color='black', alpha=0.2, label='SEM')
    
    # Add Markers
    plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='Trigger Onset')
    plt.axhline(0, color='blue', linewidth=1, linestyle='-', alpha=0.2)
    
    # Styling
    plt.title(f"Channel Index: {index} (Orig ID: {orig_id})\nDepth: {depth} $\mu m$ (Normalized)", fontsize=14)
    plt.xlabel("Time from Trigger (s)")
    plt.ylabel("Voltage ($\mu V$)")
    plt.xlim(-pre_time, post_time)
    
    # Dynamic Y-Limits (include the SEM in the bounds calculation)
    upper_bound = np.max(mean + sem)
    lower_bound = np.min(mean - sem)
    buffer = (upper_bound - lower_bound) * 0.1
    if buffer == 0: buffer = 10
    
    plt.ylim(lower_bound - buffer, upper_bound + buffer)
    
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.show()

interact(
    plot_channel, 
    index=IntSlider(
        min=0, 
        max=len(sorted_indices)-1, 
        step=1, 
        value=0, 
        description='Sort Index:',
        continuous_update=False 
    )
);




In [ ]:
import spikeinterface.extractors as se
import numpy as np
import matplotlib.pyplot as plt
import os
from ipywidgets import interact, IntSlider
import ipywidgets as widgets

# --- 1. SETUP ---
raw_path = r"Z:\Mike\Data\Psilocybin Fear Conditioning\Cohort 4_06_05_25 (SC PAG Implanted Animals)\Mouse 4\Renewal\Neural Data\Raw Data\mouse4_renewal_checkerboard"
 

# Parameters
pre_time = 0.1
post_time = 0.5
downsample_factor = 30 

# --- 2. LOAD DATA ---
rec_raw = se.read_openephys(raw_path)
rec_raw = rec_raw.rename_channels(np.arange(rec_raw.get_num_channels()))
channel_map = np.load(os.path.join(ks_path, 'channel_map.npy')).flatten()
rec = rec_raw.select_channels(channel_map)
fs = rec.get_sampling_frequency()

# --- 3. SORT CHANNELS BY DEPTH ---
chan_pos = np.load(os.path.join(ks_path, 'channel_positions.npy'))
depths = chan_pos[:, 1]
original_ids = rec.channel_ids

sorted_indices = np.argsort(depths)
sorted_depths = depths[sorted_indices]
sorted_ids = original_ids[sorted_indices]

print(f"Data loaded: {len(depths)} channels sorted by depth.")

# --- 4. PROCESS TRIGGERS (1-150) ---
subset_triggers = np.array(triggers_post[1:151], dtype=int)

if subset_triggers[0] > rec.get_num_frames():
    offset = subset_triggers[0] - (1.0 * fs) 
    local_triggers = subset_triggers - offset
else:
    local_triggers = subset_triggers

valid_mask = (local_triggers > (pre_time * fs)) & (local_triggers < (rec.get_num_frames() - post_time * fs))
final_triggers = local_triggers[valid_mask]

# --- 5. EXTRACT ALL TRIALS ---
print(f"Extracting {len(final_triggers)} trials for SEM calculation...")

num_samples = int((pre_time + post_time) * fs)
num_ds = num_samples // downsample_factor

# List to store every single trial: [ (N_ch, N_time), (N_ch, N_time), ... ]
all_trials_list = []

for trig in final_triggers:
    start = int(trig - pre_time * fs)
    end = int(trig + post_time * fs)
    trace = rec.get_traces(start_frame=start, end_frame=end).T
    
    # Downsample
    trace_ds = trace[:, ::downsample_factor]
    
    # Ensure correct shape before appending
    if trace_ds.shape[1] == num_ds:
        all_trials_list.append(trace_ds)

# Convert to 3D Array: (N_trials, N_channels, N_timepoints)
all_trials = np.array(all_trials_list)

# --- 6. BASELINE CORRECTION & STATISTICS ---
time_axis = np.linspace(-pre_time, post_time, num_ds)
zero_idx = np.abs(time_axis - 0).argmin()

# Subtract the value at t=0 for EACH trial individually
# Value at zero: (N_trials, N_channels, 1)
baseline_vals = all_trials[:, :, zero_idx][:, :, np.newaxis]
all_trials_norm = all_trials - baseline_vals

# Calculate Mean and SEM across the "Trials" axis (axis 0)
mean_trace = np.mean(all_trials_norm, axis=0) # (N_channels, N_time)
std_trace = np.std(all_trials_norm, axis=0)
sem_trace = std_trace / np.sqrt(len(all_trials_list))

# Sort by Depth
mean_sorted = mean_trace[sorted_indices, :]
sem_sorted = sem_trace[sorted_indices, :]

print("Statistics complete. Generating Widget...")

# --- 7. INTERACTIVE PLOTTER ---

def plot_channel(index):
    """
    Plots Mean +/- SEM for a single channel.
    """
    # Get data
    mean = mean_sorted[index]
    sem = sem_sorted[index]
    depth = sorted_depths[index]
    orig_id = sorted_ids[index]
    
    plt.figure(figsize=(10, 5))
    
    # Plot Mean Line
    plt.plot(time_axis, mean, color='black', linewidth=1.5, label='Mean Response')
    
    # Plot SEM Shading
    plt.fill_between(time_axis, mean - sem, mean + sem, 
                     color='black', alpha=0.2, label='SEM')
    
    # Add Markers
    plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='Trigger Onset')
    plt.axhline(0, color='blue', linewidth=1, linestyle='-', alpha=0.2)
    
    # Styling
    plt.title(f"Channel Index: {index} (Orig ID: {orig_id})\nDepth: {depth} $\mu m$ (Normalized)", fontsize=14)
    plt.xlabel("Time from Trigger (s)")
    plt.ylabel("Voltage ($\mu V$)")
    plt.xlim(-pre_time, post_time)
    
    # Dynamic Y-Limits (include the SEM in the bounds calculation)
    upper_bound = np.max(mean + sem)
    lower_bound = np.min(mean - sem)
    buffer = (upper_bound - lower_bound) * 0.1
    if buffer == 0: buffer = 10
    
    plt.ylim(lower_bound - buffer, upper_bound + buffer)
    
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.show()

interact(
    plot_channel, 
    index=IntSlider(
        min=0, 
        max=len(sorted_indices)-1, 
        step=1, 
        value=0, 
        description='Sort Index:',
        continuous_update=False 
    )
);
